# Proxima clase: ML con latencia del pipeline

Este cuaderno continua desde la practica de observabilidad. La idea es usar los datos que Spark Streaming guardo en Parquet para preparar un ejemplo sencillo de Machine Learning.

Objetivo del ejemplo:

```text
usar metricas del minuto actual -> predecir la latencia promedio del siguiente minuto
```

No usaremos Pandas ni scikit-learn. Todo sera con Spark y MLlib.

## 1. Idea general

En la sesion anterior se guardaron eventos con campos como:

- `kafkaTimestamp`: momento registrado por Kafka.
- `latencyMs`: cuanto demoro el evento entre produccion y procesamiento.
- `topic`, `partition`, `offset`: metadata Kafka.
- `origen`: productor del evento.

Para ML no trabajaremos evento por evento. Primero agruparemos la latencia por ventanas de 1 minuto.

Ejemplo:

```text
09:00 -> avgLatencyMs = 120
09:01 -> avgLatencyMs = 180
09:02 -> avgLatencyMs = 240
```

Luego intentaremos predecir la latencia promedio del siguiente minuto.

## 2. Crear SparkSession

Esta celda inicia Spark. El paquete Kafka se mantiene para que el entorno sea consistente con el cuaderno anterior, aunque aqui leeremos desde Parquet.

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("ml-latency-pipeline") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.13:4.1.1") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
spark

:: loading settings :: url = jar:file:/usr/local/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
org.apache.spark#spark-sql-kafka-0-10_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-1edb23c4-63cb-46b2-a557-2830c9faf025;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.13;4.1.1 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.13;4.1.1 in central
	found org.apache.kafka#kafka-clients;3.9.1 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.8 in central
	found org.slf4j#slf4j-api;2.0.17 in central
	found org.apache.hadoop#hadoop-client-runtime;3.4.2 in central
	found org.apache.hadoop#hadoop-client-api;3.4.2 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.scala-lang.modules#

## 3. Leer los datos guardados por streaming

Este es el punto de partida para ML. Ya no leemos directamente de Kafka; leemos el historico guardado en Parquet.

Si esta ruta no existe, primero ejecuta el cuaderno `08_observabilidad_pipeline_kafka_spark.ipynb` y genera algunos eventos.

In [2]:
# Proxima clase: ML

from pyspark.sql.functions import col, window, avg, min as spark_min, max as spark_max, count

df = spark.read.parquet("/opt/artifacts/output/orden_eventos_observabilidad_p1")

df.printSchema()
df.show(10, truncate=False)

root
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- kafkaTimestamp: timestamp (nullable = true)
 |-- tipoEvento: string (nullable = true)
 |-- ordenId: long (nullable = true)
 |-- total: double (nullable = true)
 |-- estado: string (nullable = true)
 |-- origen: string (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- isValid: boolean (nullable = false)
 |-- processedAt: long (nullable = false)
 |-- latencyMs: long (nullable = true)



+-------------+---------+------+-----------------------+------------+-------+------+---------+-----------+-------------+-------+-------------+---------+
|topic        |partition|offset|kafkaTimestamp         |tipoEvento  |ordenId|total |estado   |origen     |timestamp    |isValid|processedAt  |latencyMs|
+-------------+---------+------+-----------------------+------------+-------+------+---------+-----------+-------------+-------+-------------+---------+
|orden-eventos|0        |21    |2026-05-04 10:48:33.336|orden.creada|762    |245.0 |PENDIENTE|python     |1777891713335|true   |1777891716125|2790     |
|orden-eventos|0        |22    |2026-05-04 10:48:35.343|orden.creada|449    |488.0 |PENDIENTE|python     |1777891715342|true   |1777891716125|783      |
|orden-eventos|0        |18    |2026-05-04 09:26:15.733|orden.creada|40     |900.0 |PENDIENTE|ec-orden-ms|1777886775724|true   |1777886776046|322      |
|orden-eventos|0        |19    |2026-05-04 09:27:40.658|orden.creada|41     |1600.

## 4. Crear la serie temporal de latencia

Agrupamos los eventos por ventana de 1 minuto.

Cada fila de `serie_latency` representa un minuto del pipeline:

- `numEventos`: cuantos eventos llegaron en ese minuto.
- `avgLatencyMs`: latencia promedio del minuto.
- `minLatencyMs`: menor latencia del minuto.
- `maxLatencyMs`: mayor latencia del minuto.

Esto convierte eventos individuales en datos listos para analisis temporal.

In [3]:
serie_latency = df.groupBy(
    window(col("kafkaTimestamp"), "1 minute")
).agg(
    count("*").alias("numEventos"),
    avg("latencyMs").alias("avgLatencyMs"),
    spark_min("latencyMs").alias("minLatencyMs"),
    spark_max("latencyMs").alias("maxLatencyMs")
).select(
    col("window.start").alias("inicioVentana"),
    col("window.end").alias("finVentana"),
    "numEventos",
    "avgLatencyMs",
    "minLatencyMs",
    "maxLatencyMs"
).orderBy("inicioVentana")

serie_latency.show(100, truncate=False)

+-------------------+-------------------+----------+------------+------------+------------+
|inicioVentana      |finVentana         |numEventos|avgLatencyMs|minLatencyMs|maxLatencyMs|
+-------------------+-------------------+----------+------------+------------+------------+
|2026-05-04 09:25:00|2026-05-04 09:26:00|1         |50.0        |50          |50          |
|2026-05-04 09:26:00|2026-05-04 09:27:00|2         |554.0       |322         |786         |
|2026-05-04 09:27:00|2026-05-04 09:28:00|1         |235.0       |235         |235         |
|2026-05-04 10:48:00|2026-05-04 10:49:00|4         |1165.0      |101         |2790        |
+-------------------+-------------------+----------+------------+------------+------------+



## 5. Preparar el dataset para ML

En ML necesitamos dos cosas:

- `features`: datos conocidos que usaremos para predecir.
- `label`: valor que queremos predecir.

En este ejemplo:

```text
features = metricas del minuto actual
label = avgLatencyMs del siguiente minuto
```

Por eso creamos `nextAvgLatencyMs`, que toma la latencia promedio de la siguiente ventana.

In [4]:
from pyspark.sql.functions import lead
from pyspark.sql.window import Window

w = Window.orderBy("inicioVentana")

dataset_ml = serie_latency.withColumn(
    "nextAvgLatencyMs",
    lead("avgLatencyMs").over(w)
).filter(
    col("nextAvgLatencyMs").isNotNull()
)

dataset_ml.show(100, truncate=False)

26/05/04 20:19:52 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 20:19:52 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 20:19:52 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 20:19:53 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 20:19:53 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 20:19:53 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 2

+-------------------+-------------------+----------+------------+------------+------------+----------------+
|inicioVentana      |finVentana         |numEventos|avgLatencyMs|minLatencyMs|maxLatencyMs|nextAvgLatencyMs|
+-------------------+-------------------+----------+------------+------------+------------+----------------+
|2026-05-04 09:25:00|2026-05-04 09:26:00|1         |50.0        |50          |50          |554.0           |
|2026-05-04 09:26:00|2026-05-04 09:27:00|2         |554.0       |322         |786         |235.0           |
|2026-05-04 09:27:00|2026-05-04 09:28:00|1         |235.0       |235         |235         |1165.0          |
+-------------------+-------------------+----------+------------+------------+------------+----------------+



## 6. Convertir columnas a features

MLlib no recibe columnas separadas como entrada del modelo. Primero debemos unirlas en una sola columna llamada `features`.

Usaremos como entrada:

- `numEventos`
- `avgLatencyMs`
- `minLatencyMs`
- `maxLatencyMs`

Y como objetivo:

- `nextAvgLatencyMs`

In [5]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
    inputCols=["numEventos", "avgLatencyMs", "minLatencyMs", "maxLatencyMs"],
    outputCol="features"
)

df_features = assembler.transform(dataset_ml).select(
    "inicioVentana",
    "features",
    col("nextAvgLatencyMs").alias("label")
)

df_features.show(100, truncate=False)

26/05/04 20:20:08 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 20:20:08 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 20:20:08 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 20:20:08 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 20:20:08 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 20:20:09 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 2

+-------------------+-----------------------+------+
|inicioVentana      |features               |label |
+-------------------+-----------------------+------+
|2026-05-04 09:25:00|[1.0,50.0,50.0,50.0]   |554.0 |
|2026-05-04 09:26:00|[2.0,554.0,322.0,786.0]|235.0 |
|2026-05-04 09:27:00|[1.0,235.0,235.0,235.0]|1165.0|
+-------------------+-----------------------+------+



## 7. Entrenar una regresion lineal

La regresion lineal intenta aprender una relacion simple entre las metricas del minuto actual y la latencia del siguiente minuto.

Es un buen primer modelo porque es facil de explicar:

```text
prediccion = combinacion de numEventos, avgLatencyMs, minLatencyMs y maxLatencyMs
```

In [6]:
from pyspark.ml.regression import LinearRegression

train, test = df_features.randomSplit([0.8, 0.2], seed=42)

lr = LinearRegression(
    featuresCol="features",
    labelCol="label"
)

model = lr.fit(train)

predictions = model.transform(test)

predictions.show(100, truncate=False)

26/05/04 20:20:24 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 20:20:24 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 20:20:24 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 20:20:24 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 20:20:24 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 20:20:25 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 2

+-------------------+-----------------------+------+------------------+
|inicioVentana      |features               |label |prediction        |
+-------------------+-----------------------+------+------------------+
|2026-05-04 09:27:00|[1.0,235.0,235.0,235.0]|1165.0|450.43910258849917|
+-------------------+-----------------------+------+------------------+



## 8. Evaluar el modelo

Usaremos dos metricas:

- `RMSE`: error promedio del modelo. Mientras menor, mejor.
- `R2`: que tanto explica el modelo. Cerca de 1 es mejor; cerca de 0 indica que explica poco.

Con pocos datos, estas metricas pueden salir inestables. Eso es normal en laboratorio.

In [7]:
from pyspark.ml.evaluation import RegressionEvaluator

evaluator_rmse = RegressionEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="rmse"
)

evaluator_r2 = RegressionEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="r2"
)

rmse = evaluator_rmse.evaluate(predictions)
r2 = evaluator_r2.evaluate(predictions)

print("RMSE:", rmse)
print("R2:", r2)

26/05/04 20:20:42 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 20:20:42 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 20:20:42 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 20:20:42 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 20:20:42 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 20:20:42 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 2

RMSE: 714.5608974115008
R2: -inf


26/05/04 20:20:42 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 20:20:42 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 20:20:43 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 20:20:43 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


## 9. Como interpretar las predicciones

En la salida del modelo veras:

- `label`: latencia real del siguiente minuto.
- `prediction`: latencia estimada por el modelo.

Ejemplo:

```text
label = 250 ms
prediction = 220 ms
error aproximado = 30 ms
```

La idea no es tener un modelo perfecto, sino entender el flujo completo:

```text
datos observados -> dataset historico -> features -> modelo -> prediccion
```

## 10. Siguiente mejora

Cuando haya mas datos, se puede comparar regresion lineal contra modelos mas flexibles, por ejemplo:

- `RandomForestRegressor`
- `GBTRegressor`

Pero para iniciar, la regresion lineal es suficiente para explicar el flujo de ML distribuido con MLlib.